In [ ]:
!apt update -q && apt-get install -qy openjdk-21-jdk-headless > /dev/null 2>&1

import subprocess
from IPython.core.magic import register_cell_magic

class PersistentJShell:
    def __init__(self):
        self.proc = subprocess.Popen(
            ["jshell", "-s"],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

    def execute(self, code):
        if self.proc.poll() is not None:
            raise RuntimeError("JShell process has terminated.")

        marker = "___JSHELL_CELL_END___"
        payload = code.strip() + f'\nSystem.out.println("{marker}");\n'
        self.proc.stdin.write(payload)
        self.proc.stdin.flush()

        lines = []
        while True:
            line = self.proc.stdout.readline()
            if not line:
                break
            if marker in line:
                break
            lines.append(line)

        return "".join(lines)

_jshell_instance = PersistentJShell()

@register_cell_magic
def jshell(line, cell):
    out = _jshell_instance.execute(cell)
    if out:
        print(out, end="")


In [ ]:
%%jshell

void runPy(String code) throws Exception {
    String marker = "___END_OF_PYTHON_CELL___";

    String payload = """
        exec('''%s''')
        print('%s')
        """.formatted(code.replace("'", "\\'"), marker);

    pyIn.println(payload);

    String line;
    while ((line = pyOut.readLine()) != null) {
        if (line.contains(marker)) break;
        System.out.println(line);
    }
}


# Lesson 7: vLLM SageMaker Realtime Deployment
This notebook demonstrates setting up AWS credentials via Colab Secrets, creating standard S3 buckets, and configuring a vLLM SageMaker Realtime Endpoint.

In [ ]:
%%jshell

runPy("""
# Cell 1 - Setup Environment & Repo
os.system('pip install boto3 sagemaker');

from pathlib import Path
import os

REPO_ROOT = Path('/content/BITS_programming')
if not REPO_ROOT.exists():
os.system('git clone https://github.com/aqwertyuiop48/BITS_programming.git /content/BITS_programming');

NOTEBOOK_DIR = REPO_ROOT / 'module_5/week_27'
os.chdir(NOTEBOOK_DIR)
print(f'Working directory: {NOTEBOOK_DIR}')
""");


In [ ]:
%%jshell

runPy("""
# Cell 2 - Retrieve AWS Credentials safely via Colab Secrets / Env
import os

def get_colab_secret(*names, required=True):
    """Read the first available credential from env vars or Colab Secrets."""
    for name in names:
        value = os.environ.get(name)
        if value:
            return value

    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    if userdata is not None:
        for name in names:
            try:
                value = os.environ.get(name) or userdata.get(name)
                if value:
                    return value
            except Exception:
                continue

    if required:
        choices = " or ".join(names)
        raise RuntimeError(
            f"Missing AWS credential. Add a Colab Secret named {choices} and enable notebook access, "
            f"or set one of those environment variables."
        )
    return None

# Fetch AWS credentials from Colab secrets
AWS_ACCESS_KEY_ID = get_colab_secret('AWS_ACCESS_KEY_ID', 'AWS_ACCESS_KEY')
AWS_SECRET_ACCESS_KEY = get_colab_secret('AWS_SECRET_ACCESS_KEY', 'AWS_SECRET_KEY')
AWS_SESSION_TOKEN = get_colab_secret('AWS_SESSION_TOKEN', 'AWS_SECURITY_TOKEN', required=False)

os.environ['AWS_ACCESS_KEY_ID'] = AWS_ACCESS_KEY_ID
os.environ['AWS_SECRET_ACCESS_KEY'] = AWS_SECRET_ACCESS_KEY
if AWS_SESSION_TOKEN:
    os.environ['AWS_SESSION_TOKEN'] = AWS_SESSION_TOKEN

print("AWS credentials loaded from Colab secrets and set as environment variables.")
""");


In [ ]:
%%jshell

runPy("""
# Block 1 - Config & S3 Bucket Naming
import boto3
import datetime as dt
from botocore.exceptions import ClientError

AWS_REGION_NAME     = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
ALLOW_AWS_MUTATIONS = True

PROJECT_NAME  = "lesson7-vllm"
ENVIRONMENT   = "demo"
HF_MODEL_ID   = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
NUM_GPUS      = 1

os.environ["AWS_DEFAULT_REGION"] = AWS_REGION_NAME
os.environ["AWS_REGION"]         = AWS_REGION_NAME
region       = AWS_REGION_NAME

boto_session = boto3.Session(region_name=region)
sts = boto_session.client("sts")
_ident = sts.get_caller_identity()
account_id = _ident["Account"]
caller_arn = _ident["Arn"]

s3  = boto_session.client("s3")
sm  = boto_session.client("sagemaker")
smr = boto_session.client("sagemaker-runtime")
iam = boto_session.client("iam")
cw  = boto_session.client("cloudwatch")

# Dynamic S3 Bucket setup following standardization
S3_BUCKET_NAME = f"{PROJECT_NAME}-{account_id}-{region}"
MODEL_S3_URI  = f"s3://{S3_BUCKET_NAME}/{PROJECT_NAME}/model/"

# Ensure standard bucket exists
try:
    if region == "us-east-1":
        s3.create_bucket(Bucket=S3_BUCKET_NAME)
    else:
        s3.create_bucket(
            Bucket=S3_BUCKET_NAME,
            CreateBucketConfiguration={'LocationConstraint': region}
        )
    print(f"Target S3 Bucket ready: s3://{S3_BUCKET_NAME}")
except ClientError as e:
    if e.response['Error']['Code'] in ['BucketAlreadyOwnedByYou', 'BucketAlreadyExists']:
        print(f"Using existing S3 Bucket: s3://{S3_BUCKET_NAME}")
    else:
        raise e

GPU_INSTANCE_CANDIDATES = [
    "ml.g4dn.xlarge",
    "ml.g5.xlarge",
    "ml.g4dn.2xlarge",
    "ml.g5.2xlarge",
    "ml.p3.2xlarge",
]

INSTANCE_HOURLY = {
    "ml.g4dn.xlarge": 0.736, "ml.g5.xlarge": 1.408, "ml.g4dn.2xlarge": 0.94,
    "ml.g5.2xlarge": 1.515, "ml.p3.2xlarge": 3.825,
}

timestamp       = dt.datetime.utcnow().strftime("%Y%m%d-%H%M%S")
model_name      = f"{PROJECT_NAME}-{ENVIRONMENT}-model-{timestamp}"
endpoint_name   = f"{PROJECT_NAME}-{ENVIRONMENT}-ep"
endpoint_config = None
chosen_instance = None

print(f"Region:     {region}")
print(f"Account:    {account_id}")
print(f"Model:      {HF_MODEL_ID}")
print(f"S3 URI:     {MODEL_S3_URI}")
print(f"Sweep:      {GPU_INSTANCE_CANDIDATES}")
print(f"Endpoint:   {endpoint_name}")
""");


In [ ]:
%%jshell

runPy("""
import datetime, pytz;
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))
""");
